# Notebook to test out dimensionality reduction techniques. Show metrics in loss of MSE, L1 and Cosine Similarity

In [2]:
import torch
from torch.utils.data import DataLoader, Dataset, Subset
import einops
from sklearn.decomposition import PCA
from data.dataset import BEVFeaturesDataset
from sklearn import config_context
import sklearn

print('The scikit-learn version is {}.'.format(sklearn.__version__))

The scikit-learn version is 1.0.2.


In [3]:
def load_data(channels=256):
    # Load the saved data
    dataset = BEVFeaturesDataset(root_dir='/home/mingdayang/FeatureBridgeMapping/data/bev_features', transform=None, channels=channels)

    return dataset

def create_splits(dataset, train_split=0.8):
    gen = torch.Generator()
    gen.manual_seed(0)
    train_dataset, test_dataset = torch.utils.data.random_split(dataset, [int(train_split * len(dataset)), len(dataset) - int(train_split * len(dataset))], generator=gen)

    return train_dataset, test_dataset

def make_loader(batch_size, dataset):

    dataloader = DataLoader(dataset, batch_size=batch_size, shuffle=True)
        # Let's check out what we've created

    return dataloader

batch_size = 1


dataset = load_data(channels=256)
dataset_10_samples = Subset(dataset, [0, 1, 2, 3, 4, 5, 6, 7, 8, 9])
train_dataset, test_dataset = create_splits(dataset)
train_loader = make_loader(batch_size, train_dataset)
test_loader = make_loader(batch_size, test_dataset)
single_loader = DataLoader(Subset(dataset, [0]), batch_size=batch_size, shuffle=False)
print(len(train_dataset), len(test_dataset))


64 17


In [4]:
amount_of_samples = len(dataset)
img_tensor, bev_tensor = torch.zeros((amount_of_samples, 256, 200, 200)), torch.zeros((amount_of_samples, 256, 200, 200))

for i, (img, bev) in enumerate(dataset):
    img_tensor[i] = dataset[i][img]
    bev_tensor[i] = dataset[i][bev]
print(img_tensor.shape, bev_tensor.shape)

torch.Size([81, 256, 200, 200]) torch.Size([81, 256, 200, 200])


In [5]:
# Normalize using z-score normalization
img_mean = img_tensor.mean(dim=0)
img_std = img_tensor.std(dim=0)
bev_mean = bev_tensor.mean(dim=0)
bev_std = bev_tensor.std(dim=0)

def normalize(tensor, mean, std):
    return (tensor - mean) / std

def denormalize(tensor, mean, std):
    return tensor * std + mean

img_tensor_normalized = normalize(img_tensor, img_mean, img_std)
bev_tensor_normalized = normalize(bev_tensor, bev_mean, bev_std)

In [6]:

device='cpu'
img_tensor_samples_features = einops.rearrange(img_tensor_normalized, 'b c w h -> c (b w h)').to(device)
bev_tensor_samples_features = einops.rearrange(bev_tensor_normalized, 'b c w h -> c (b w h)').to(device)
print(img_tensor_samples_features.shape, bev_tensor_samples_features.shape)

torch.Size([256, 3240000]) torch.Size([256, 3240000])


In [7]:

pca_img_99 = PCA(n_components=0.99, svd_solver='full')
pca_bev_99 = PCA(n_components=0.99, svd_solver='full')

pca_img_95 = PCA(n_components=0.95, svd_solver='full')
pca_bev_95 = PCA(n_components=0.95, svd_solver='full')

# PCA fit expects array X (n_samples, n_features)
pca_img_99.fit(img_tensor_samples_features)
pca_bev_99.fit(bev_tensor_samples_features)
pca_img_95.fit(img_tensor_samples_features)
pca_bev_95.fit(bev_tensor_samples_features) 

img_99_data = pca_img_99.transform(img_tensor_samples_features)
bev_99_data = pca_bev_99.transform(bev_tensor_samples_features)
img_95_data = pca_img_95.transform(img_tensor_samples_features)
bev_95_data = pca_bev_95.transform(bev_tensor_samples_features)

print(f"Number of components to retain 99% variance for image features: {pca_img_99.n_components_}")
print(f"Number of components to retain 99% variance for bev features: {pca_bev_99.n_components_}")
print(f"Number of components to retain 95% variance for image features: {pca_img_95.n_components_}")
print(f"Number of components to retain 95% variance for bev features: {pca_bev_95.n_components_}")

Number of components to retain 99% variance for image features: 187
Number of components to retain 99% variance for bev features: 213
Number of components to retain 95% variance for image features: 91
Number of components to retain 95% variance for bev features: 130
